In [0]:
%run /Repos/awproject/databricks_learning/config/config


In [0]:
# =============================================================================
# Notebook: 00_batch_control
# Path:     /Repos/awproject/databricks_learning/config/00_batch_control
# Purpose:  Single control table for all layers — bronze, silver, gold
# Usage:    %run /Repos/awproject/databricks_learning/config/00_batch_control
# Depends:  00_config must be run first
# =============================================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

PIPELINE_SCHEMA     = f"{CATALOG}.pipeline"
BATCH_CONTROL_TABLE = f"{PIPELINE_SCHEMA}.batch_control"


# =============================================================================
# INIT
# =============================================================================

def init_batch_control():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PIPELINE_SCHEMA}")

    if not spark.catalog.tableExists(BATCH_CONTROL_TABLE):
        spark.sql(f"""
            CREATE TABLE {BATCH_CONTROL_TABLE} (
                id                   BIGINT GENERATED ALWAYS AS IDENTITY,
                table_name           STRING    NOT NULL,
                layer                STRING    NOT NULL,
                load_type            STRING    NOT NULL,
                source_type          STRING    NOT NULL,
                watermark_col        STRING,
                last_load_timestamp  TIMESTAMP,
                batch_start_time     TIMESTAMP,
                batch_end_time       TIMESTAMP,
                status               STRING,
                rows_loaded          STRING
            )
            USING DELTA
        """)
        print(f"  Created: {BATCH_CONTROL_TABLE}")

    # Seed if empty — handles both first creation and post-TRUNCATE
    row_count = spark.read.table(BATCH_CONTROL_TABLE).count()
    if row_count == 0:
        spark.sql(f"""
            INSERT INTO {BATCH_CONTROL_TABLE}
                (table_name, layer, load_type, source_type, watermark_col,
                 last_load_timestamp, batch_start_time, batch_end_time,
                 status, rows_loaded)
            VALUES
                -- ── BRONZE — source_type = file (CSV from ADLS) ────────────
                -- file sources are always FULL — no watermark needed
                ('calendar',              'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),
                ('customers',             'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),
                ('product_categories',    'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),
                ('product_subcategories', 'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),
                ('products',              'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),
                ('territories',           'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),
                ('sales',                 'bronze', 'FULL', 'file', NULL, NULL, NULL, NULL, NULL, NULL),

                -- ── SILVER — source_type = delta (reading from bronze) ──────
                ('calendar',              'silver', 'FULL', 'delta', NULL,        NULL, NULL, NULL, NULL, NULL),
                ('customers',             'silver', 'FULL', 'delta', NULL,        NULL, NULL, NULL, NULL, NULL),
                ('product_categories',    'silver', 'FULL', 'delta', NULL,        NULL, NULL, NULL, NULL, NULL),
                ('product_subcategories', 'silver', 'FULL', 'delta', NULL,        NULL, NULL, NULL, NULL, NULL),
                ('products',              'silver', 'FULL', 'delta', NULL,        NULL, NULL, NULL, NULL, NULL),
                ('territories',           'silver', 'FULL', 'delta', NULL,        NULL, NULL, NULL, NULL, NULL),
                ('sales',                 'silver', 'INCR', 'delta', 'OrderDate', NULL, NULL, NULL, NULL, NULL),

                -- ── GOLD — source_type = delta (reading from silver) ────────
                ('dim_date',             'gold', 'FULL', 'delta', NULL,         NULL, NULL, NULL, NULL, NULL),
                ('dim_territory',        'gold', 'FULL', 'delta', NULL,         NULL, NULL, NULL, NULL, NULL),
                ('dim_product_category', 'gold', 'FULL', 'delta', NULL,         NULL, NULL, NULL, NULL, NULL),
                ('dim_product',          'gold', 'FULL', 'delta', NULL,         NULL, NULL, NULL, NULL, NULL),
                ('dim_customer',         'gold', 'FULL', 'delta', NULL,         NULL, NULL, NULL, NULL, NULL),
                ('fact_sales',           'gold', 'INCR', 'delta', 'order_date', NULL, NULL, NULL, NULL, NULL)
        """)
        print(f"  Seeded 20 rows across bronze + silver + gold")
    else:
        print(f"  Batch control ready: {BATCH_CONTROL_TABLE} ({row_count} rows)")


# =============================================================================
# READ — get last run info for a table
# =============================================================================

def get_last_load(table_name: str, layer: str) -> dict:
    """
    Returns load config for a table.
    source_type drives logic in each layer notebook:
      file  → always FULL, no watermark
      delta → FULL or INCR, watermark applies for INCR
      db    → FULL or INCR, watermark applies for INCR
    """
    rows = (
        spark.read.table(BATCH_CONTROL_TABLE)
        .filter(
            (F.col("table_name") == table_name) &
            (F.col("layer")      == layer)
        )
        .orderBy(F.col("batch_end_time").desc_nulls_last())
        .limit(1)
        .collect()
    )
    if not rows:
        return {
            "id":                   None,
            "load_type":            "FULL",
            "source_type":          "file",
            "watermark_col":        None,
            "last_load_timestamp":  None,
            "status":               None,
        }

    row     = rows[0]
    last_ts = row["last_load_timestamp"]
    last_ts_str = last_ts.strftime("%Y-%m-%d") if last_ts else None

    return {
        "id":                   row["id"],
        "load_type":            row["load_type"],
        "source_type":          row["source_type"],
        "watermark_col":        row["watermark_col"],
        "last_load_timestamp":  last_ts_str,
        "status":               row["status"],
    }


def get_tables_by_layer(layer: str) -> list:
    """Returns all tables for a layer ordered by id."""
    rows = (
        spark.read.table(BATCH_CONTROL_TABLE)
        .filter(F.col("layer") == layer)
        .orderBy("id")
        .select("id", "table_name", "layer", "load_type", "source_type", "watermark_col")
        .collect()
    )
    return [row.asDict() for row in rows]


# =============================================================================
# LOG START
# =============================================================================

def log_batch_start(table_name: str, layer: str, load_type: str) -> None:
    """Call at the start of each table load."""
    DeltaTable.forName(spark, BATCH_CONTROL_TABLE).alias("t").merge(
        spark.createDataFrame(
            [(table_name, layer, load_type)],
            ["table_name", "layer", "load_type"]
        ).alias("s"),
        "t.table_name = s.table_name AND t.layer = s.layer"
    ).whenMatchedUpdate(set={
        "batch_start_time": "current_timestamp()",
        "batch_end_time":   "null",
        "status":           "'RUNNING'",
        "rows_loaded":      "null",
    }).execute()
    print(f"  [RUNNING] {table_name} ({layer})")


# =============================================================================
# LOG END
# =============================================================================

def log_batch_end(table_name: str, layer: str,
                  watermark, rows_loaded: int,
                  status: str = "SUCCESS") -> None:
    """Call at the end of each table load — success or failure."""
    if watermark:
        from datetime import datetime
        wm_str = str(watermark).strip()
        for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S"):
            try:
                wm_str = datetime.strptime(wm_str[:len(fmt)], fmt).strftime("%Y-%m-%d")
                break
            except ValueError:
                continue
        wm_expr = f"to_timestamp('{wm_str}', 'yyyy-MM-dd')"
    else:
        wm_expr = "null"

    DeltaTable.forName(spark, BATCH_CONTROL_TABLE).alias("t").merge(
        spark.createDataFrame(
            [(table_name, layer)],
            ["table_name", "layer"]
        ).alias("s"),
        """
        t.table_name     = s.table_name
        AND t.layer      = s.layer
        AND t.status     = 'RUNNING'
        AND t.batch_end_time IS NULL
        """
    ).whenMatchedUpdate(set={
        "last_load_timestamp": wm_expr,
        "batch_end_time":      "current_timestamp()",
        "status":              f"'{status}'",
        "rows_loaded":         f"'{rows_loaded}'",
    }).execute()

    print(f"  [{status}] {table_name} ({layer}) — {rows_loaded:,} rows | watermark: {watermark}")


# =============================================================================
# SHOW
# =============================================================================

def show_control_table(layer: str = None):
    """
    show_control_table()           → all layers
    show_control_table('bronze')   → bronze only
    show_control_table('gold')     → gold only
    """
    df = spark.read.table(BATCH_CONTROL_TABLE)
    if layer:
        df = df.filter(F.col("layer") == layer)
    df.orderBy("layer", "id") \
      .select(
          "id", "layer", "table_name", "source_type", "load_type",
          "watermark_col", "status", "rows_loaded",
          "last_load_timestamp", "batch_start_time", "batch_end_time"
      ).show(truncate=False)


# =============================================================================
# INITIALISE ON EVERY %run
# =============================================================================
init_batch_control()
print(f"Batch control loaded → {BATCH_CONTROL_TABLE}")